# 01. NV1: Kiểm soát chất lượng và Tiền xử lý dữ liệu (Dung làm)

## Mục 2.2 trong Báo cáo Đồ án Business Analyst

Trước khi phân tích, dữ liệu phiên sạc cần được kiểm tra và tiền xử lý nhằm bảo đảm tính hợp lệ, nhất quán của các thông tin sử dụng. Quy trình tập trung kiểm tra dữ liệu thiếu, bản ghi trùng lặp, quan hệ giữa tổng thời gian kết nối và thời gian sạc, đồng thời đánh giá tính hợp lý của điện năng ghi nhận. Các giá trị bất thường được nhận diện bằng kiểm tra phân phối và phương pháp khoảng tứ phân vị (IQR), sau đó đối chiếu với quy tắc nghiệp vụ để lựa chọn cách xử lý phù hợp. 

Những phiên có thời gian kết nối dài không bị loại bỏ chỉ vì vượt ngưỡng thống kê, bởi đây có thể là biểu hiện của hành vi chiếm dụng mà đề tài cần nghiên cứu. Dữ liệu gốc cần được giữ nguyên và các bước xử lý được ghi lại để thuận tiện kiểm tra, đối chiếu. Qua đó, dữ liệu được chuẩn bị cho việc xây dựng đặc trưng, phân tích hành vi, đánh giá hiệu suất và dự báo nhu cầu sử dụng trạm sạc.

**Đầu ra sau NV1:**
- `data/processed/sessions_clean.parquet`: Bộ dữ liệu đã tiền xử lý, gắn đầy đủ cờ chất lượng.
- `outputs/tables/data_quality_audit_summary.csv`: Bảng kiểm toán chất lượng dữ liệu.


In [ ]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Thêm thư mục gốc vào sys.path để import src
sys.path.append(str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from src.config import (
    RAW_DATA_FILE, 
    SESSIONS_CLEAN_PARQUET, 
    QUALITY_AUDIT_CSV,
    FIGURES_DIR
)
from src.data_validation import (
    check_duplicates,
    check_duration_consistency,
    check_physical_power_limits,
    detect_port_overlaps,
    compute_iqr_fences,
    generate_data_quality_report
)
from src.feature_engineering import (
    parse_duration_to_seconds,
    parse_dates_robust,
    convert_to_utc,
    compute_idle_metrics,
    build_clean_pipeline
)
from src.visualization import (
    set_plotting_style,
    plot_distributions,
    plot_boxplots,
    plot_power_anomalies
)

set_plotting_style()
print("Các module và thư viện đã được nạp thành công.")


### 2.2.1. Đọc dữ liệu và Kiểm tra toàn vẹn logic nghiệp vụ
Ký hiệu tổng thời gian kết nối là $T_{total}$, thời gian sạc là $T_{charging}$ và điện năng cung cấp là $E$. Các quy tắc kiểm tra bao gồm:
1. **Tính đầy đủ:** Không mặc định giá trị thiếu là 0.
2. **Trình tự thời gian:** $T_{end} \ge T_{start}$ sau khi quy đổi UTC.
3. **Quan hệ thời lượng:** $T_{total} > 0$ và $0 \le T_{charging} \le T_{total}$. Gắn cờ 45 dòng vi phạm, không tự động gán idle time âm về 0.
4. **Quan hệ điện năng:** $E \ge 0$. Nếu $E > 0 \Rightarrow T_{charging} > 0$.
5. **Tính duy nhất:** Loại bỏ 4 bản ghi trùng lặp hoàn toàn.
6. **Tính nhất quán cổng sạc:** Định danh bằng `MAC Address` + `Port Number`, kiểm tra 181 phiên chồng lấn.


In [ ]:
# Đọc dữ liệu thô
print(f"Đọc dữ liệu từ: {RAW_DATA_FILE}")
df_raw = pd.read_csv(RAW_DATA_FILE, low_memory=False)
print(f"Kích thước ban đầu: {len(df_raw):,} dòng, {len(df_raw.columns)} cột")

# Thực thi pipeline làm sạch dữ liệu
df_clean, initial_count, dup_count = build_clean_pipeline(df_raw)
print(f"Số dòng sau loại trùng: {len(df_clean):,}")
print(f"Số dòng vi phạm T_charging > T_total: {df_clean['flag_charging_gt_total'].sum():,}")
print(f"Số dòng vi phạm giới hạn công suất vật lý: {df_clean['flag_physical_power_anomaly'].sum():,}")
print(f"Số dòng chồng lấn cổng sạc vật lý: {df_clean['flag_port_overlap'].sum():,}")
print(f"Số phiên hợp lệ đầy đủ cho phân tích: {df_clean['is_valid_session'].sum():,}")


### 2.2.2. Phân tích ngoại lệ thống kê qua Hàng rào IQR
$$IQR = Q_3 - Q_1$$
$$L = Q_1 - 1.5 \times IQR, \quad U = Q_3 + 1.5 \times IQR$$
*Lưu ý nghiệp vụ:* Không tự động winsorize hay xóa bỏ các phiên sạc thời lượng dài vì đây là hành vi chiếm dụng hoặc sạc qua đêm cần nghiên cứu.


In [ ]:
# Thống kê IQR trên các phiên hợp lệ logic
valid_logic = df_clean[~df_clean['flag_charging_gt_total']]

for col, name in [('total_duration_hours', 'Tổng thời gian kết nối (h)'),
                  ('charging_time_hours', 'Thời gian sạc (h)'),
                  ('Energy (kWh)', 'Điện năng tiêu thụ (kWh)')]:
    fences = compute_iqr_fences(valid_logic[col])
    exceed = (valid_logic[col] > fences['Upper_Fence']).sum()
    print(f"{name:30s} | Q1={fences['Q1']:.3f} | Q3={fences['Q3']:.3f} | U={fences['Upper_Fence']:.3f} | Vượt U: {exceed:,}")


In [ ]:
# Trực quan hóa phân phối và biểu đồ hộp
plot_distributions(df_clean, show=True)
plot_boxplots(df_clean, show=True)


### 2.2.3. Kiểm tra tính hợp lệ vật lý và trần công suất
$$P_{avg} = \frac{E}{T_{charging}} \quad (\text{kW})$$
Phát hiện bản ghi dị biệt tại `PALO ALTO CA / TED THOMPSON #1` (Port Type Level 2, CT4020-HD-GW) ghi nhận $E = 0.699\text{ kWh}$ trong $6\text{ giây} \Rightarrow P_{avg} = 419.4\text{ kW}$.


In [ ]:
# Định vị bản ghi dị biệt công suất kinh điển
anom_row = df_clean[(df_clean['charging_time_sec'] == 6) & (df_clean['Energy (kWh)'] == 0.699)]
display(anom_row[['Station Name', 'Port Type', 'Model Number', 'Total Duration (hh:mm:ss)', 'Charging Time (hh:mm:ss)', 'Energy (kWh)', 'p_avg_kw']])

# Biểu đồ phân phối công suất sạc
plot_power_anomalies(df_clean, show=True)


### 2.2.4. Phân lập Zero-Energy Sessions và Lưu trữ kết quả NV1


In [ ]:
# Bảng kiểm toán chất lượng dữ liệu
audit_report = generate_data_quality_report(df_clean, initial_count, dup_count)
display(audit_report)

# Lưu trữ file dữ liệu sạch parquet
df_clean.to_parquet(SESSIONS_CLEAN_PARQUET, index=False, engine='pyarrow')
print(f"Đã lưu thành công dữ liệu sạch tại: {SESSIONS_CLEAN_PARQUET}")
